<a href="https://colab.research.google.com/github/joaovitorpr/cdscience/blob/main/Estudo_CP1_nemec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('/content/dataset_delivery_fraude_avaliacoes.csv')
df.columns = [c.replace('ï»¿', '') for c in df.columns]
df.head()

#Fase.1
media = df['preco_pedido'].mean() #Tira a media
mediana = df['preco_pedido'].median() #Tira a mediana
limpeza = df['preco_pedido'].dropna() #Retirada dos valores nulos
media_aparada = stats.trim_mean(limpeza, proportiontocut=0.1) #Tirando a media aparada

print(media)
print(mediana)
print(media_aparada)

#Respostas
#1)Sim.
#2)Sim, a evidencia de outliers. Pois, os valores estão muito distoantes um do outro.
#3)Usaria a mediana.Devido a mesma ser mais robusta em seu resultado, pois a mesma não altera seu valor mesmo com a presença de outliers. Perfeita para lidar com valores de esquemas fraudulentos.

#Fase.2
media_ponderada = df['nota_cliente'] * df['preco_pedido'] #Multiplicação dos pesos com o valores(preco)
media_ponderada = media_ponderada.sum() / df['preco_pedido'].sum() #soma de todas os pesos e valores divido pela soma total dos valores(precos)

print(media_ponderada)
print(df['nota_cliente'].mean())
#Respostas:
#1)Não, pois em comparação com a media de notas. A quantidade de avaliações baixas para os outros restaurantes é menor. O que significa que os usuarios estão insatisfeitos de acordo com o que pagaram.
#2)Diria que essa percepção é um vies. Pois os dois tem uma correlação quase inexistente.

#Fase.3

#1)
#Quantile descobre o percentil
q1 = df['tempo_entrega_min'].quantile(0.25) #Descobre o percentil 25
q3 = df['tempo_entrega_min'].quantile(0.75) #Descobre o percentil 75

#2)
IQR = q3 - q1 #Descobre o IQR

#3)
limite_inferio = q1 - 1.5 * IQR #Descobre o limite inferior
limite_superior = q3 + 1.5 * IQR #Descobre o limite superior

print(f"Q1: {q1}")
print(f"Q3:{q3}")
print(f"IQR:{IQR}")
print(f"limite_inferio:{limite_inferio}")
print(f"limite_superior:{limite_superior}")

#Respostas
#1)
outliers = df['tempo_entrega_min'][(df['tempo_entrega_min'] < limite_inferio) | (df['tempo_entrega_min'] > limite_superior)].tolist() #Descobre os outliers
print(f"Os outliers são: {outliers} ")

#2)Sim. Pois o valor da media em comparação a mediana é maior, demonstrando que há um leve impacto.
#3)Analisando em visão de vies, sim. Mas em questão real, não. Pois eles seriam importantes para vermos quais esquemas seriam fraudulentos ou não.

#Fase.4
percentil_25 = df['preco_pedido'].quantile(0.25)
percentil_50 = df['preco_pedido'].quantile(0.50)
percentil_75 = df['preco_pedido'].quantile(0.75)

percentil_90 = df['preco_pedido'].quantile(0.90)

mais_caros = df[df['preco_pedido'] > percentil_90]
print(mais_caros)

#O que defini um pedido caro seria um pedido cujo o valor está acima de 75% e mais caro quando acima de 90%.

#Fase.5
#1
ct_real = df['categoria'].unique()
for ct in ct_real:
    ctf = df[df['categoria'] == ct].copy()
    ct_total = len(ctf)
    if ct_total == 0:
        continue

    q1 = ctf['taxa_entrega'].quantile(0.25)
    q3 = ctf['taxa_entrega'].quantile(0.75)

    def classificar(taxa):
      if taxa > q3:
        return "Alta"
      elif taxa < q1:
        return "Baixa"
      else:
        return "Média"

    ctf['classe_taxa'] = ctf['taxa_entrega'].apply(classificar)
    p_alta = len(ctf[ctf['classe_taxa'] == "Alta"]) / ct_total
    p_baixa = len(ctf[ctf['classe_taxa'] == "Baixa"]) / ct_total
    p_media = len(ctf[ctf['classe_taxa'] == "Média"]) / ct_total

    media_taxa = ctf['taxa_entrega'].mean()

    R_alta = 1.5 * media_taxa
    R_media = 1 * media_taxa
    R_baixa = 0.5 * media_taxa

    EV = (p_alta * R_alta) + (p_media * R_media) + (p_baixa * R_baixa)

    print(f"Categoria: {ct} | EV: R$ {EV:.2f}")

#


#Fase.6

correlacao_pearson = df['nota_cliente'].corr(df['nota_cliente'])
correlacao_pearson2 = df['preco_pedido'].corr(df['nota_cliente'])
correlacao_pearson3 = df['distancia_km'].corr(df['tempo_entrega_min'])

print(correlacao_pearson)
print(correlacao_pearson2)
print(correlacao_pearson3)

#Não. Nem sempre uma correlação vai dizer que algo está ligado diretamente a uma coisa ou acontecimento.

#Fase.7

rest = df.groupby(['restaurante_id', 'nome_estabelecimento']).agg(
    n=('pedido_id', 'count'),
    nota_media=('nota_cliente', 'mean'),
    nota_std=('nota_cliente', 'std'),
    preco_medio=('preco_pedido', 'mean'),
    dist_media=('distancia_km', 'mean'),
    taxa_media=('taxa_entrega', 'mean'),
).reset_index()

suspeitos = rest.sort_values(['nota_media', 'nota_std'], ascending=[False, True])
print(suspeitos.head(5))

r18 = df[df['restaurante_id'] == 'R018']
print(r18[['item_principal','preco_pedido','distancia_km','taxa_entrega','nota_cliente']].to_string())
print(r18['item_principal'].value_counts())





72.31084
54.11
62.33219999999999
3.689308850512593
3.6966
Q1: 30.0
Q3:53.0
IQR:23.0
limite_inferio:-4.5
limite_superior:87.5
Os outliers são: [164, 217, 189, 119, 133, 144, 147, 193, 172, 114, 91, 176, 118, 142] 
    pedido_id restaurante_id            nome_estabelecimento   categoria  \
4       P0005           R001  Armazém da Coxinha Clandestino       Doces   
6       P0007           R001  Armazém da Coxinha Clandestino       Doces   
8       P0009           R001  Armazém da Coxinha Clandestino       Doces   
51      P0052           R005      Armazém do Abacaxi Honesto     Marmita   
58      P0059           R006       Armazém do Cometa Sagrado  Hambúrguer   
..        ...            ...                             ...         ...   
944     P0945           R095           Vila da Garoa Honesto     Marmita   
965     P0966           R097         Vila do Abacaxi Sincero  Hambúrguer   
966     P0967           R097         Vila do Abacaxi Sincero  Hambúrguer   
991     P0992           R10